Building the metadata dataframe starts with inspecting the StopWhitefly monitoring page and identifying its embedded iframes. The page contains three iframe URLs corresponding to the interactive trap map, the overall weekly average whitefly-count plot, and the degree-day plot. For metadata extraction, we focus on the first iframe: the EDDMapS trap map. This map contains the trap/site markers, and each marker represents a monitoring location with an internal EDDMapS site_id, a human-readable label, latitude, longitude, status, and report-date information.

To understand where the marker data come from, we request the map iframe HTML using Python requests with a browser-like User-Agent header. We then inspect the returned HTML to determine whether the site metadata are directly embedded in the HTML or loaded later by JavaScript. The HTML does not contain a clean hard-coded metadata table. Instead, it contains JavaScript that calls `loadClustering()` with a relative AJAX endpoint: `data.cfc?method=relativecatch&` plus query-string parameters.

The next step is to reconstruct the full AJAX endpoint URL. The original map iframe URL contains query parameters such as sub, proj, project, lat, lng, zoom, and map-display options. Using Python’s `urlparse()`, we extract the query string from the map iframe URL. Using `urljoin()`, we resolve the relative endpoint data.cfc?method=relativecatch&... against the map iframe path. This produces the full metadata endpoint URL.

When we request this endpoint, it returns JSON. The JSON object contains three important components: records, columns, and count. The count value tells us how many marker records were returned. The columns list gives the field names, such as SITEID, SITENAME, LAT, LON, STATUS, FIRSTREPORT, and LASTREPORT. The records object is a list of row values, where each inner list corresponds to one trap/site marker. Because the response provides both columns and records, we can safely construct a pandas dataframe by using records as the rows and columns as the column names.

Then we clean the raw metadata dataframe by keeping only fields whose meanings are clear and useful for downstream extraction: SITEID, SITENAME, LAT, LON, STATUS, FIRSTREPORT, and LASTREPORT. We rename these into analysis-friendly names: site_id, trap_label, latitude, longitude, status, first_report, and last_report. We convert the report-date fields to datetime values. We do not need to derive a separate numeric trap_id because site_id is the stable machine-facing identifier used by the EDDMapS endpoints. The final metadata table has one row per monitoring site and can later be merged with weekly trap-count observations using site_id.

`The map iframe is a discovery interface: its HTML reveals an AJAX endpoint that`
`returns marker metadata as JSON, and that JSON gives us the site_id values and` 
`coordinates needed to drive later weekly-count extraction.`

### Step 1: Is the map iframe just HTML, or does it contain clues?

When we fetch a web page with Python, we usually get the initial HTML source. But 
modern pages often load data later using JavaScript. So, our first question is 
very basic- 

`When Python requests the map iframe URL, does the returned HTML contain usueful 
clues about where the marker/trap metadata comes from?`

In [1]:
import requests

In [3]:
map_url = (
    "https://maps.eddmaps.org/point/trap/relativecatch/index.cfm?"
    "sub=10378&proj=1441&project=1441&lat=31.35&lng=-83.60&zoom=10.5"
    "&map=136&shownodata=1&maxzoom=12&circlemarker&expiredata=14"
    )

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    )
}

response = requests.get(map_url, headers=headers, timeout=30)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("HTML length:", len(response.text))

Status code: 200
Content-Type: text/html;charset=UTF-8
HTML length: 25958


If `status_code` is 200, Python successfully fetched the map iframe page.

Search through the HTML text chunks to investigate what kind and form of information 
is available in the text. Goal is to determine if the data is available directly 
inside HTML, or it points to external data sources.

Lots of `<html>`, `<script>`, `<link>`    -->  indicates a web page shell

Many rows of repeated trap/location values  -->  Data may be embedded directly

JavaScript files like .js   -->   Logic may be stored in external scripts

Function names    -->    JavaScript may be building the map

URLs or partial URLs   -->   Possible hidden data requests

Words like `ajax`, `getJSON`, `fetch`, `data`   -->   Page may request data after loading

In [ ]:
print(response.text[12000:14000])

       name: 'Cluster',
                source: clusterSource,
                animationDuration:  200,
                // Cluster style
                style: getStyle
            });
            map.addLayer(clusterLayer);

            NProgress.configure({ minimum: 0.001,  trickle: false });
            NProgress.start();

            
                loadClustering("data.cfc?method=relativecatch&"+getquerystring());
            
            function loadClustering(url) {
                $.ajax({
                    url: url,
                    type: 'GET',
                    dataType: 'json', 
                    crossDomain: true,
                    xhrFields: { withCredentials: true },
                    success: function(data) {
                        $.each(data.records, function(i,x) {
                            var px = x[13]; // longitude
                            var py = x[12]; // latitude
                            var point = new ol.geom.Point(ol.proj.fromLonLat

The top 4000 characters contain lots of `<html>`, `<scripts>`, `<links>`, urls to .js files.

From 4000 characters onward, I see iframe map embeddings with URLs from maps.eddmaps.org

After 6000, I see functions having words like 'query_string'

In characters 10,000 to 12,000, I see words like lat = '31.35' and lng = '-83.60'

In characters 12,000 to 14,000, I see functions like loadClustering(url) and things like- 
loadClustering("data.cfc?method=relativecatch&"+getquerystring());
```javascript
function loadClustering(url) {
                $.ajax({
                    url: url,
                    type: 'GET',
                    dataType: 'json', 
                    crossDomain: true,
                    xhrFields: { withCredentials: true },
                    success: function(data) {
                        $.each(data.records, function(i,x) {
                            var px = x[13]; // longitude
                            var py = x[12]; // latitude
                            var point = new ol.geom.Point(ol.proj.fromLonLat([px, py]));

                            /*Status function*/
                            if (x[14] == 'NoData') {
                                var status = 'No Data';
                            } else if (x[14] == 'OldData') {
                                var status = 'Inactive Site';
                            } else if (x[8] >= '1' && x[8] <= '1') {var status = 'Low';} else if (x[8] >= '2' && x[8] <= '2') {var status = 'Low-Med';} else if (x[8] >= '3' && x[8] <= '3') {var status = 'Med-High';} else if (x[8] >= '4' && x[8] <= '4') {var status = 'High';} else  {
                                return;
                            }
                            
                            var iconFeature = new ol.Feature({
                                geometry: point,
                                name: status,
                                description : x,
                                type: 'click'
                            }
```

#### What I discovered?

The HTML page itself is mostly a map application shell. It contains scripts and 
map setup code, but the actual trap/site marker data are not hard-coded as a clean 
table in the first few thousand characters.

#### What the page says?

`When the map loads, call loadClustering() and ask another URL for marker data.`

The other URL is- "data.cfc?method=relativecatch& + getquerystring()"

#### So now we have a new hypothesis:

The map iframe loads trap/site metadata through an AJAX request to 
`data.cfc?method=relativecatch`, using the same query parameters from the iframe URL.

#### What AJAX means here?

It means JavaScript running in the browser makes a separate request for data after 
the original HTML page has loaded.

#### This part is especially important:
```javascript
$.each(data.records, function(i,x) {
```

It tells us that each x is one record, probably one trap/site marker.

These lines- 
```javascript
var px = x[13]; // longitude
var py = x[12]; // latitude
```

tells us that the records are probably list-like rows, where column position matters.

#### Next small step

We need to reconstruct the exact URL that the browser would call.



### Use Python to extract the query string from the map URL

In [19]:
from urllib.parse import urlparse

parsed = urlparse(map_url)

print("Scheme:", parsed.scheme)
print("Domain:", parsed.netloc)
print("Path:", parsed.path)
print("Query string:")
print(parsed.query)

Scheme: https
Domain: maps.eddmaps.org
Path: /point/trap/relativecatch/index.cfm
Query string:
sub=10378&proj=1441&project=1441&lat=31.35&lng=-83.60&zoom=10.5&map=136&shownodata=1&maxzoom=12&circlemarker&expiredata=14


### Reconstruct the likely AJAX endpoint URL

In [21]:
from urllib.parse import urljoin

metadata_url = urljoin(
    map_url,
    "data.cfc?method=relativecatch&" + parsed.query
)

print(metadata_url)

https://maps.eddmaps.org/point/trap/relativecatch/data.cfc?method=relativecatch&sub=10378&proj=1441&project=1441&lat=31.35&lng=-83.60&zoom=10.5&map=136&shownodata=1&maxzoom=12&circlemarker&expiredata=14


`urljoin(base, relative)` follows browser rules:

If `base` ends with a file, the file is replaced.

If `base` ends with a slash, the relative path is appended.

### Request the metadata endpoint and inspect the response

In [22]:
metadata_response = requests.get(metadata_url, headers=headers, timeout=30)

print("Status code:", metadata_response.status_code)
print("Content-Type:", metadata_response.headers.get("Content-Type"))
print("Response length:", len(metadata_response.text))
print(metadata_response.text[:1000])

Status code: 200
Content-Type: application/json;charset=UTF-8
Response length: 4092
{"records":[[29221,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 101",31.46603,-83.465008,"Active","April, 22 2020 00:00:00"],[29222,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 102",31.465087,-83.405028,"Active","April, 22 2020 00:00:00"],[29224,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 104",31.557827,-83.480183,"Active","April, 22 2020 00:00:00"],[29225,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 105",31.541213,-83.57304,"Active","April, 22 2020 00:00:00"],[29226,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 106",31.524275,-83.662878,"Active","April, 22 2020 00:00:00"],[29227,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 107",31.45218,-83.614588,"Active","April, 22 2020 00:00:00"],[29228,2,7,3,1,4,27,1.0,2,12,"June, 02 2026 00:00:00","Trap 108",31.381952,-83.630712,"Active","April, 22 2020 00:00:00"],[29229,0,7,1,1,1,1,0.0,1,12,"June, 02 2026 00:00:00","Trap 

`records` is a list of rows. Each inner list is one trap/site marker.

Currently, we don't have column names to convert the records into a pandas dataframe. 
So next, we will parse the JSON to check if column names are separately provided.

### Parse the JSON and inspect its structure

In [23]:
metadata = metadata_response.json()

print(type(metadata))
print(metadata.keys())

<class 'dict'>
dict_keys(['records', 'columns', 'count'])


In [27]:
print("Count:", metadata.get("count"))

print("\nColumns:")
print(metadata.get("columns"))

print("\nNumber of records:", len(metadata.get("records")))

print("\nFirst record:")
print(metadata["records"][0])

Count: 31

Columns:
['SITEID', 'TOTALQTY', 'CATCHPERIOD', 'DENSERANK', 'TRAPS', 'QUARTILE', 'RANKS', 'PERCENTRANK', 'CUSTOMRANK', 'DAYSSINCE', 'LASTREPORT', 'SITENAME', 'LAT', 'LON', 'STATUS', 'FIRSTREPORT']

Number of records: 31

First record:
[29221, 0, 7, 1, 1, 1, 1, 0.0, 1, 12, 'June, 02 2026 00:00:00', 'Trap 101', 31.46603, -83.465008, 'Active', 'April, 22 2020 00:00:00']


### Build the first raw dataframe

Now we can safely make a dataframe using `records` and `columns`

In [28]:
import pandas as pd

metadata_raw_df = pd.DataFrame(
    metadata["records"],
    columns=metadata["columns"]
)

print(metadata_raw_df.shape)
metadata_raw_df.head()

(31, 16)


,SITEID,TOTALQTY,CATCHPERIOD,DENSERANK,TRAPS,QUARTILE,RANKS,PERCENTRANK,CUSTOMRANK,DAYSSINCE,LASTREPORT,SITENAME,LAT,LON,STATUS,FIRSTREPORT
0,29221,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,12.0,"June, 02 2026 00:00:00",Trap 101,31.466030,-83.465008,Active,"April, 22 2020 00:00:00"
1,29222,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,12.0,"June, 02 2026 00:00:00",Trap 102,31.465087,-83.405028,Active,"April, 22 2020 00:00:00"
2,29224,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,12.0,"June, 02 2026 00:00:00",Trap 104,31.557827,-83.480183,Active,"April, 22 2020 00:00:00"
3,29225,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,12.0,"June, 02 2026 00:00:00",Trap 105,31.541213,-83.573040,Active,"April, 22 2020 00:00:00"
4,29226,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,12.0,"June, 02 2026 00:00:00",Trap 106,31.524275,-83.662878,Active,"April, 22 2020 00:00:00"


In [29]:
metadata_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SITEID       31 non-null     int64  
 1   TOTALQTY     27 non-null     float64
 2   CATCHPERIOD  27 non-null     float64
 3   DENSERANK    27 non-null     float64
 4   TRAPS        27 non-null     float64
 5   QUARTILE     27 non-null     float64
 6   RANKS        27 non-null     float64
 7   PERCENTRANK  27 non-null     float64
 8   CUSTOMRANK   27 non-null     float64
 9   DAYSSINCE    27 non-null     float64
 10  LASTREPORT   27 non-null     str    
 11  SITENAME     31 non-null     str    
 12  LAT          31 non-null     float64
 13  LON          31 non-null     float64
 14  STATUS       31 non-null     str    
 15  FIRSTREPORT  31 non-null     str    
dtypes: float64(11), int64(1), str(4)
memory usage: 4.0 KB


### Create a clean metadata dataframe

In [30]:
metadata_clean_df = metadata_raw_df[
    [
        "SITEID",
        "SITENAME",
        "LAT",
        "LON",
        "STATUS",
        "FIRSTREPORT",
        "LASTREPORT",
    ]
].copy()

metadata_clean_df.head()

,SITEID,SITENAME,LAT,LON,STATUS,FIRSTREPORT,LASTREPORT
0,29221,Trap 101,31.466030,-83.465008,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
1,29222,Trap 102,31.465087,-83.405028,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
2,29224,Trap 104,31.557827,-83.480183,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
3,29225,Trap 105,31.541213,-83.573040,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
4,29226,Trap 106,31.524275,-83.662878,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"


### Rename the columns into Python-friendly snake_case:

In [31]:
metadata_clean_df = metadata_clean_df.rename(
    columns={
        "SITEID": "site_id",
        "SITENAME": "trap_label",
        "LAT": "latitude",
        "LON": "longitude",
        "STATUS": "status",
        "FIRSTREPORT": "first_report",
        "LASTREPORT": "last_report",
    }
)

metadata_clean_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
1,29222,Trap 102,31.465087,-83.405028,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
2,29224,Trap 104,31.557827,-83.480183,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
3,29225,Trap 105,31.541213,-83.573040,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
4,29226,Trap 106,31.524275,-83.662878,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"


### Convert report dates

Right now `first_report` and `last_report` are strings. Convert them to real 
datetimes:

In [35]:
metadata_clean_df["first_report"] = pd.to_datetime(
    metadata_clean_df["first_report"],
    errors="coerce"
)

metadata_clean_df["last_report"] = pd.to_datetime(
    metadata_clean_df["last_report"],
    errors="coerce"
)

metadata_clean_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-02
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-02
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-02
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-02
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-02


### Validate the clean metadata table

In [36]:
metadata_clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   site_id       31 non-null     int64         
 1   trap_label    31 non-null     str           
 2   latitude      31 non-null     float64       
 3   longitude     31 non-null     float64       
 4   status        31 non-null     str           
 5   first_report  31 non-null     datetime64[us]
 6   last_report   27 non-null     datetime64[us]
dtypes: datetime64[us](2), float64(2), int64(1), str(2)
memory usage: 1.8 KB


In [37]:
metadata_clean_df[metadata_clean_df["last_report"].isna()]

,site_id,trap_label,latitude,longitude,status,first_report,last_report
25,31565,Shiver Rd.,31.001615,-84.215862,OldData,2025-04-23,NaT
26,31566,Melton Brinson Rd.,30.867297,-84.359697,OldData,2025-04-23,NaT
29,32409,Morven Field,30.933030,-83.479394,OldData,2025-08-21,NaT
30,32410,Dixie Field,30.798771,-83.688641,OldData,2025-08-21,NaT


### Save the clean metadata table

In [41]:
from pathlib import Path

output_path = Path("../data/interim/trap_metadata.csv")

metadata_clean_df.to_csv(output_path, index=False)